# mIF Pipeline Debug Notebook

This notebook is a notebook-first wrapper around the Python API in `src/mif_pipeline`.

Use it to:
- inspect resolved config paths
- dry-run the full slide plan
- run one stage at a time and rerun only the failing stage
- inspect outputs between steps

Real data paths are expected to be cluster paths, so the dry-run cells are useful locally.

Set `REPO_ROOT` explicitly in the next code cell before running the rest of the notebook.

Important setup note: `setup_slide(...)` writes a starter channel map to `setup.channel_map_output`. If you want downstream steps in this notebook session to use that generated file immediately, flip `USE_GENERATED_CHANNEL_MAP = True` below and rerun the config-loading cell after the setup cell has written the file.


In [1]:
# Set this to your checkout of the repo before running the notebook.
REPO_ROOT = "/home/ratnayn/codex/mIF-pipeline/"


In [2]:
from pathlib import Path
import json
import sys
import traceback

if "REPO_ROOT" not in globals():
    raise RuntimeError(
        "Set REPO_ROOT to the repository path before running this cell. Example: REPO_ROOT = '/abs/path/to/mIF-pipeline'"
    )

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError(
        f"REPO_ROOT does not look like the repo root: {REPO_ROOT}. Expected to find {REPO_ROOT / 'src'}."
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Source dir added to sys.path: {SRC_DIR}")


Repo root: /home/ratnayn/codex/mIF-pipeline
Source dir added to sys.path: /home/ratnayn/codex/mIF-pipeline/src


In [3]:
from mif_pipeline import (
    export_masks,
    load_config,
    merge_slide_ometiffs,
    qc_slide,
    run_all,
    run_instanseg,
    run_nimbus_chunked,
    setup_slide,
)
from mif_pipeline.config import get_slide_config, resolve_channel_entries, resolve_nimbus_inputs
from mif_pipeline.export_masks import mask_stems_for_slide
from mif_pipeline.instanseg_runner import instanseg_zarr_path


def show(data):
    print(json.dumps(data, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{label} failed: {exc}")
        traceback.print_exc()
        raise


In [4]:
CONFIG_PATH ="~/codex/mIF-pipeline/prototyping/prototype_v2.yaml"
SLIDE_ID = "SLIDE-0329"

FORCE = False
RUN_DRY = False

# If setup_slide writes a starter map and you want this notebook session
# to use it immediately, flip this to True and rerun the next cell.
USE_GENERATED_CHANNEL_MAP = True

print(f"CONFIG_PATH = {CONFIG_PATH}")
print(f"SLIDE_ID = {SLIDE_ID}")
print(f"FORCE = {FORCE}")
print(f"RUN_DRY = {RUN_DRY}")
print(f"USE_GENERATED_CHANNEL_MAP = {USE_GENERATED_CHANNEL_MAP}")


CONFIG_PATH = ~/codex/mIF-pipeline/prototyping/prototype_v2.yaml
SLIDE_ID = SLIDE-0329
FORCE = False
RUN_DRY = False
USE_GENERATED_CHANNEL_MAP = True


In [5]:
config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)

if USE_GENERATED_CHANNEL_MAP:
    generated_map = slide.get("setup", {}).get("channel_map_output")
    if not generated_map:
        raise ValueError(
            "USE_GENERATED_CHANNEL_MAP is True but setup.channel_map_output is missing."
        )
    config["slides"][SLIDE_ID]["channel_map_file"] = generated_map
    slide = get_slide_config(config, SLIDE_ID)

nimbus_inputs = resolve_nimbus_inputs(config, SLIDE_ID)
seg_entries = resolve_channel_entries(
    config, SLIDE_ID, slide.get("seg_merge", {}).get("channels", [])
)
full_entries = resolve_channel_entries(
    config, SLIDE_ID, slide.get("full_merge", {}).get("channels", [])
)
mask_stems = mask_stems_for_slide(config, SLIDE_ID, image_paths=nimbus_inputs["fov_paths"])

ome_path = Path(slide["seg_merge"]["ome_path"])
zarr_path = instanseg_zarr_path(ome_path, slide["instanseg"]["prediction_tag"])

summary = {
    "slide_id": SLIDE_ID,
    "slide_dir": slide["slide_dir"],
    "output_dir": slide["output_dir"],
    "channel_map_file": slide["channel_map_file"],
    "setup_channel_map_output": slide.get("setup", {}).get("channel_map_output"),
    "seg_merge_inputs": [entry["path"] for entry in seg_entries],
    "full_merge_inputs": [entry["path"] for entry in full_entries],
    "nimbus_raw_paths": nimbus_inputs["raw_paths"],
    "nimbus_fov_paths": nimbus_inputs["fov_paths"],
    "mask_stems": mask_stems,
    "segmentation_ome_path": str(ome_path),
    "instanseg_zarr_path": str(zarr_path),
}
show(summary)


{
  "slide_id": "SLIDE-0329",
  "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329",
  "output_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs",
  "channel_map_file": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
  "setup_channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
  "seg_merge_inputs": [
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.1_R000_FITC_AF_I.ome.tif",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.1_R000_Cy3_AF_I.ome.tif",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.1_R000_Cy5_AF_I.ome.tif",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.1_R000_Cy7_AF_I.ome.tif"
  ],
  "full_merge_inputs": [],
  "nimbus_raw_paths": [
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.4_R000_DAPI__FINAL_F.ome.tif",
    "/mnt/c

In [6]:
def import_status(module_name):
    try:
        __import__(module_name)
        return "OK"
    except Exception as exc:
        return f"MISSING: {exc}"


dependency_status = {
    "yaml": import_status("yaml"),
    "tifffile": import_status("tifffile"),
    "ome_types": import_status("ome_types"),
    "numpy": import_status("numpy"),
    "skimage": import_status("skimage"),
    "zarr": import_status("zarr"),
    "instanseg": import_status("instanseg"),
    "tiffslide": import_status("tiffslide"),
    "nimbus_inference": import_status("nimbus_inference"),
    "pandas": import_status("pandas"),
}
show(dependency_status)


{
  "yaml": "OK",
  "tifffile": "OK",
  "ome_types": "OK",
  "numpy": "OK",
  "skimage": "OK",
  "zarr": "OK",
  "instanseg": "OK",
  "tiffslide": "OK",
  "nimbus_inference": "OK",
  "pandas": "OK"
}


## Dry-Run Planning

Leave `RUN_DRY = True` while you validate config resolution, output paths, and chunk planning.


In [7]:
dry_run_plan = stage_call("run_all(dry_run=True)", run_all, config, SLIDE_ID, dry_run=True)


=== run_all(dry_run=True) ===
{
  "slide_id": "SLIDE-0329",
  "dry_run": true,
  "setup": {
    "slide_id": "SLIDE-0329",
    "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329",
    "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
    "channel_patterns": [
      "*.tif",
      "*.tiff",
      "*.ome.tif",
      "*.ome.tiff"
    ],
    "channel_count": 24,
    "aliases": [
      "R1_CY3_AF_I",
      "R1_CY5_AF_I",
      "R1_CY7_AF_I",
      "R1_FITC_AF_I",
      "R1_BIT2_RS0584_CY3B",
      "R1_BIT3_RS0015_CY5",
      "R1_BIT4_RS0083_750",
      "R1_DAPI",
      "R1_BIT1_RS0996_488",
      "R2_BIT6_RS0639_CY3B",
      "R2_BIT7_RS0109_CY5",
      "R2_BIT8_RS0255_750",
      "R2_DAPI",
      "R2_BIT5_RS1047_488",
      "R3_BIT10_RS0763_CY3B",
      "R3_BIT11_RS1312_CY5",
      "R3_BIT12_RS0237_750",
      "R3_DAPI",
      "R3_BIT9_RS0805_488",
      "R4_P19_POLYRAT",
      "R4_ANXA10_647",
      "R4_LGALS4_750",
      "R4_DAPI",
 

## Run Individual Stages

Set `RUN_DRY = False` above when you are on the cluster and ready to execute a real step.

These are split into separate cells so you can rerun only the stage that failed.


In [8]:
if slide.get("setup"):
    setup_result = stage_call(
        f"setup_slide(dry_run={RUN_DRY})",
        setup_slide,
        config,
        SLIDE_ID,
        force=FORCE,
        dry_run=RUN_DRY,
    )
else:
    print("No setup block configured for this slide.")


=== setup_slide(dry_run=False) ===
{
  "slide_id": "SLIDE-0329",
  "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329",
  "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json",
  "channel_patterns": [
    "*.tif",
    "*.tiff",
    "*.ome.tif",
    "*.ome.tiff"
  ],
  "channel_count": 24,
  "aliases": [
    "R1_CY3_AF_I",
    "R1_CY5_AF_I",
    "R1_CY7_AF_I",
    "R1_FITC_AF_I",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8_RS0255_750",
    "R2_DAPI",
    "R2_BIT5_RS1047_488",
    "R3_BIT10_RS0763_CY3B",
    "R3_BIT11_RS1312_CY5",
    "R3_BIT12_RS0237_750",
    "R3_DAPI",
    "R3_BIT9_RS0805_488",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750",
    "R4_DAPI",
    "R4_GFP_POLY_AF488"
  ],
  "status": "skipped",
  "dry_run": false
}


In [9]:
merge_result = stage_call(
    f"merge_slide_ometiffs(dry_run={RUN_DRY})",
    merge_slide_ometiffs,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


=== merge_slide_ometiffs(dry_run=False) ===
[merge] starting seg_merge for SLIDE-0329: 5 channels -> /mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome.tif
[merge] writing SLIDE-0329_segment_merge.ome.tif: 5 channels, 8 pyramid levels
[merge] channel 1/5 R1_DAPI: read level 0, rebuilding pyramid
[merge] channel 1/5 R1_DAPI: wrote level 0
[merge] channel 1/5 R1_DAPI: wrote pyramid level 1/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 2/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 3/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 4/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 5/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 6/7
[merge] channel 1/5 R1_DAPI: wrote pyramid level 7/7
[merge] channel 2/5 R1_FITC_AF_I: reading level 0
[merge] channel 2/5 R1_FITC_AF_I: rebuilding pyramid
[merge] channel 2/5 R1_FITC_AF_I: wrote level 0
[merge] channel 2/5 R1_FITC_AF_I: wrote pyramid level 1/7
[merge] channel 2/5 R1_FITC_AF_I: wrote pyramid leve

In [9]:
instanseg_result = stage_call(
    f"run_instanseg(dry_run={RUN_DRY})",
    run_instanseg,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


=== run_instanseg(dry_run=False) ===
Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/../bioimageio_models/, loading
Requesting default device: cuda


/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/inference_class.py:649: UserWarning: The image pixel size None is not in microns.
  warnings.warn("The image pixel size {} is not in microns.".format(img_pixel_size))
Slide progress:   1%|▌                                                                  | 5/575 [00:06<12:24,  1.31s/it]/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/pytorch_utils.py:286: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  sparse_onehot = torch.sparse_coo_tensor(torch.stack((zz, xxyy)).long(), (torch.ones_like(xxyy).float()),
/home/rat

Reached maximum number of iterations - this is not expected!


Slide progress:  74%|████████████████████████████████████████████████▍                | 428/575 [20:15<05:16,  2.15s/it]

Reached maximum number of iterations - this is not expected!


Slide progress: 100%|█████████████████████████████████████████████████████████████████| 575/575 [25:56<00:00,  2.71s/it]

{
  "slide_id": "SLIDE-0329",
  "ome_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome.tif",
  "zarr_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome_instanseg_prediction.zarr",
  "model": "fluorescence_nuclei_and_cells",
  "tile_size": 2000,
  "overlap": 100,
  "dry_run": false,
  "status": "written"
}


In [10]:
export_result = stage_call(
    f"export_masks(dry_run={RUN_DRY})",
    export_masks,
    config,
    SLIDE_ID,
    image_paths=nimbus_inputs["fov_paths"],
    force=FORCE,
    dry_run=RUN_DRY,
)


=== export_masks(dry_run=False) ===
{
  "slide_id": "SLIDE-0329",
  "ome_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome.tif",
  "zarr_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/SLIDE-0329_segment_merge.ome_instanseg_prediction.zarr",
  "mask_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/masks",
  "mask_stems": [
    "SLIDE-0329"
  ],
  "dry_run": false,
  "status": "written",
  "target_shape": [
    62617,
    66406
  ],
  "cell_masks": [
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/masks/SLIDE-0329_whole_cell.tiff"
  ],
  "nuclear_masks": [
    "/mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/masks/SLIDE-0329_nuclear.tiff"
  ]
}


In [8]:
nimbus_result = stage_call(
    f"run_nimbus_chunked(dry_run={RUN_DRY})",
    run_nimbus_chunked,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


=== run_nimbus_chunked(dry_run=False) ===


/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/utils.py:10: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329'].
Channels: ['SLIDE-0329_1.0.4_R000_DAPI__FINAL_F'].
Iterate over fovs...
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329/outputs/nimbus/chunk_000
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329...


  0%|          | 0/1 [00:00<?, ?it/s]

Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt


  0%|          | 0/760 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
qc_result = stage_call("qc_slide", qc_slide, config, SLIDE_ID)


## Full Run Shortcut

After debugging individual stages, you can use the single `run_all(...)` wrapper below.


In [9]:
run_all_result = stage_call(
    f"run_all(dry_run={RUN_DRY})",
    run_all,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)


=== run_all(dry_run=False) ===


/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/inference_class.py:649: UserWarning: The image pixel size None is not in microns.
  warnings.warn("The image pixel size {} is not in microns.".format(img_pixel_size))


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/../bioimageio_models/, loading
Requesting default device: cuda


Slide progress:   0%|▏                                                                 | 1/400 [00:00<06:28,  1.03it/s]/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/pytorch_utils.py:286: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  sparse_onehot = torch.sparse_coo_tensor(torch.stack((zz, xxyy)).long(), (torch.ones_like(xxyy).float()),
/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/instanseg/utils/pytorch_utils.py:312: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to

Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048'].
Channels: ['SLIDE-0329_1.0.4_R000_DAPI__FINAL_F'].
Iterate over fovs...
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/nimbus/chunk_000
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048...


...

Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048'].
Channels: ['SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F'].
Iterate over fovs...
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/nimbus/chunk_002
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048...


  0%|          | 0/1 [00:00<?, ?it/s]

Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048'].
Channels: ['SLIDE-0329_4.0.4_R000_Cy5_Anxa10-647_FINAL_AFR_F'].
Iterate over fovs...
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/nimbus/chunk_003
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048...


  0%|          | 0/1 [00:00<?, ?it/s]

{
  "slide_id": "SLIDE-0329_crop_2048",
  "dry_run": false,
  "setup": {
    "slide_id": "SLIDE-0329_crop_2048",
    "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
    "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/channel_map.generated.json",
    "channel_patterns": [
      "*.tif",
      "*.tiff",
      "*.ome.tif",
      "*.ome.tiff"
    ],
    "channel_count": 24,
    "aliases": [
      "R1_CY3_AF_I",
      "R1_CY5_AF_I",
      "R1_CY7_AF_I",
      "R1_FITC_AF_I",
      "R1_BIT2_RS0584_CY3B",
      "R1_BIT3_RS0015_CY5",
      "R1_BIT4_RS0083_750",
      "R1_DAPI",
      "R1_BIT1_RS0996_488",
      "R2_BIT6_RS0639_CY3B",
      "R2_BIT7_RS0109_CY5",
      "R2_BIT8_RS0255_750",
      "R2_DAPI",
      "R2_BIT5_RS1047_488",
      "R3_BIT10_RS0763_CY3B",
      "R3_BIT11_RS1312_CY5",
      "R3_BIT12_RS0237_750",
      "R3_DAPI",
      "R3_BIT9_RS0805_488",
      "R4_P19_POLYRAT",
      "R4_ANXA10_647",
      "R4_LGALS4_750",
      "